<a href="https://colab.research.google.com/github/kannanrk28/Capstone_Masai_FinalProject_Kannan/blob/main/part3/Part3_AdvanceModeling_Ensembles_Tuning_FullMLWorkflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

First, mount your Google Drive so this notebook can access files stored there.

In [ ]:
# Part 3 - Pre-requisite : Load cleaned_data.csv from Part 1
# Feature Matrix, Regression Label, Classification label

import pandas as pd

# Load cleaned dataset
URL = "https://raw.githubusercontent.com/kannanrk28/Capstone_Masai_FinalProject_Kannan/main/dataset/cleaned_data.csv"
df_clean = pd.read_csv(URL)
# df_clean = pd.read_csv("cleaned_data.csv")

# -----------------------------------------
# Define target variables
# -----------------------------------------

# Regression target: continuous revenue value
y_reg = df_clean["revenue"]

# Classification target: 1 = above median revenue, 0 = below/equal median revenue
y_clf = (df_clean["revenue"] > df_clean["revenue"].median()).astype(int)

# -----------------------------------------
# Drop unwanted / leakage / noisy features
# -----------------------------------------
drop_cols = [
    'revenue',                 # Target variable
    'revenue_adj',             # Data leakage (derived from revenue)
    'id',                      # Noisy feature (unique identifier)
    'imdb_id',                 # Noisy feature (unique movie identifier)
    'homepage',                # Noisy feature (URL, mostly unique and many missing values)
    'overview',                # Noisy feature (free-text description)
    'tagline',                 # Noisy feature (free-text field)
    'keywords',                # Noisy feature (text with high-cardinality keywords)
    'cast',                    # Noisy feature (high-cardinality text)
    'original_title',          # Noisy feature (mostly unique movie titles)
    'popularity',              # Potential data leakage (calculated after movie release)
    'vote_average',            # Potential data leakage (available after audience ratings)
    'vote_count',              # Potential data leakage (generated after movie release)
    'budget_adj'               # Redundant feature (inflation-adjusted version of budget)
]

# Convert release_date to datetime
df_clean['release_date'] = pd.to_datetime(df_clean['release_date'])

# Extract only the month
df_clean['release_month'] = df_clean['release_date'].dt.month

# Drop release_date after extracted date since year column is available and month is extracted. Date has no value for this usecase
df_clean.drop(columns=['release_date'], inplace=True)

X = df_clean.drop(columns=drop_cols)

print("Feature Matrix Shape :", X.shape)
print("Regression Target Shape :", y_reg.shape)
print("Classification Target Shape :", y_clf.shape)

# Display class distribution
print("\nClassification Target Distribution:")
print(y_clf.value_counts())
print(X.dtypes)

In [ ]:
# Part 3 - Pre-requisite : Encoding Categorical columns
import pandas as pd

# Create a copy of the feature matrix to avoid changing the original X
X_encoded = X.copy()

# One-hot encode genres because it has multiple values separated by |
genre_dummies = X_encoded['genres'].str.get_dummies(sep='|')

# Drop categorical columns that are not suitable for one-hot encoding
# director and production_companies have high cardinality, so encoding them would create too many columns
drop_cols = ['genres', 'director', 'production_companies']

# Drop unwanted categorical columns and add encoded genre columns
X_encoded = pd.concat(
    [X_encoded.drop(columns=drop_cols), genre_dummies],
    axis=1
)

# Display encoded feature matrix information
print("Shape of Encoded Feature Matrix:")
print(X_encoded.shape)

print("\nData Types:")
print(X_encoded.dtypes)

print("\nFirst Five Rows:")
print(X_encoded.head())

In [ ]:
# Part3 - pre-requisite: Leak Free, Train, Test, Split and Scaling

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# -----------------------------------------
# Single Train-Test Split for both tasks
# -----------------------------------------
X_train, X_test, y_train_reg, y_test_reg, y_train_clf, y_test_clf = train_test_split(
    X_encoded,
    y_reg,
    y_clf,
    test_size=0.2,
    random_state=42
)

# -----------------------------------------
# Feature Scaling
# -----------------------------------------
scaler = StandardScaler()

# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using same scaler
X_test_scaled = scaler.transform(X_test)

# -----------------------------------------
# Verify Shapes
# -----------------------------------------
print("Training Feature Shape :", X_train_scaled.shape)
print("Testing Feature Shape  :", X_test_scaled.shape)

print("\nRegression Training Target Shape :", y_train_reg.shape)
print("Regression Testing Target Shape  :", y_test_reg.shape)

print("\nClassification Training Target Shape :", y_train_clf.shape)
print("Classification Testing Target Shape  :", y_test_clf.shape)

In [ ]:
# Part 3 - Task 1 : Decision Tree baseline:

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Create Decision Tree model with default settings
dt_model = DecisionTreeClassifier(random_state=42)

# Train the model
dt_model.fit(X_train_scaled, y_train_clf)

# Predict on training and test data
y_train_pred_dt = dt_model.predict(X_train_scaled)
y_test_pred_dt = dt_model.predict(X_test_scaled)

# Calculate accuracy
train_accuracy = accuracy_score(y_train_clf, y_train_pred_dt)
test_accuracy = accuracy_score(y_test_clf, y_test_pred_dt)

print("Decision Tree Baseline Performance")
print(f"Training Accuracy : {train_accuracy:.4f}")
print(f"Test Accuracy     : {test_accuracy:.4f}")

In [ ]:
# Part 3 - Task 2 : Control Decision Tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# ==========================================
# Controlled Decision Tree
# ==========================================

controlled_dt = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

# Train the model
controlled_dt.fit(X_train_scaled, y_train_clf)

# Predictions
y_train_pred = controlled_dt.predict(X_train_scaled)
y_test_pred = controlled_dt.predict(X_test_scaled)

# Accuracy
train_accuracy = accuracy_score(y_train_clf, y_train_pred)
test_accuracy = accuracy_score(y_test_clf, y_test_pred)

print("Controlled Decision Tree Performance")
print(f"Training Accuracy : {train_accuracy:.4f}")
print(f"Test Accuracy     : {test_accuracy:.4f}")

In [ ]:
# Part 3 - Task 3 :Gini vs Entropy Comparison

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# ==========================================
# Decision Tree using Gini Index
# ==========================================

gini_tree = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    random_state=42
)

gini_tree.fit(X_train_scaled, y_train_clf)

gini_pred = gini_tree.predict(X_test_scaled)

gini_accuracy = accuracy_score(y_test_clf, gini_pred)

# ==========================================
# Decision Tree using Entropy
# ==========================================

entropy_tree = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=5,
    random_state=42
)

entropy_tree.fit(X_train_scaled, y_train_clf)

entropy_pred = entropy_tree.predict(X_test_scaled)

entropy_accuracy = accuracy_score(y_test_clf, entropy_pred)

# ==========================================
# Comparison Table
# ==========================================

comparison = pd.DataFrame({
    "Criterion": ["Gini", "Entropy"],
    "Test Accuracy": [gini_accuracy, entropy_accuracy]
})

print("Decision Tree: Gini vs Entropy")
print(comparison)

In [ ]:
# Part 3 - Task 4: Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd

# ==========================================
# Random Forest Classifier
# ==========================================

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

# Train the model
rf_model.fit(X_train_scaled, y_train_clf)

# Predictions
y_train_pred = rf_model.predict(X_train_scaled)
y_test_pred = rf_model.predict(X_test_scaled)

# Prediction probabilities
y_test_prob = rf_model.predict_proba(X_test_scaled)[:, 1]

# ==========================================
# Model Evaluation
# ==========================================

train_accuracy = accuracy_score(y_train_clf, y_train_pred)
test_accuracy = accuracy_score(y_test_clf, y_test_pred)
roc_auc = roc_auc_score(y_test_clf, y_test_prob)

print("Random Forest Performance")
print(f"Training Accuracy : {train_accuracy:.4f}")
print(f"Test Accuracy     : {test_accuracy:.4f}")
print(f"ROC-AUC           : {roc_auc:.4f}")

# ==========================================
# Feature Importance
# ==========================================

feature_importance = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Importance": rf_model.feature_importances_
})

# Sort in descending order
feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop 5 Important Features")
print(feature_importance.head(5))

In [ ]:
# Part 3 - Task 4a - Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd

# ==========================================
# Gradient Boosting Classifier
# ==========================================

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Train the model
gb_model.fit(X_train_scaled, y_train_clf)

# Predictions
y_train_pred_gb = gb_model.predict(X_train_scaled)
y_test_pred_gb = gb_model.predict(X_test_scaled)

# Prediction probabilities
y_test_prob_gb = gb_model.predict_proba(X_test_scaled)[:, 1]

# ==========================================
# Model Evaluation
# ==========================================

train_accuracy = accuracy_score(y_train_clf, y_train_pred_gb)
test_accuracy = accuracy_score(y_test_clf, y_test_pred_gb)
roc_auc = roc_auc_score(y_test_clf, y_test_prob_gb)

print("Gradient Boosting Performance")
print(f"Training Accuracy : {train_accuracy:.4f}")
print(f"Test Accuracy     : {test_accuracy:.4f}")
print(f"ROC-AUC           : {roc_auc:.4f}")

In [ ]:
# Part 3- Task 4b - Feature ablation study
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd

# ==========================================
# Identify 5 Least Important Features
# ==========================================

least_important = feature_importance.sort_values(
    by="Importance",
    ascending=True
).head(5)

print("Five Least Important Features")
print(least_important)

least_features = least_important["Feature"].tolist()

# ==========================================
# Remove Least Important Features
# ==========================================

X_train_reduced = X_train.drop(columns=least_features)
X_test_reduced = X_test.drop(columns=least_features)

# Scale reduced dataset
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_reduced_scaled = scaler.fit_transform(X_train_reduced)
X_test_reduced_scaled = scaler.transform(X_test_reduced)

# ==========================================
# Train Reduced Random Forest
# ==========================================

rf_reduced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_reduced.fit(X_train_reduced_scaled, y_train_clf)

# Predict probabilities
y_prob_reduced = rf_reduced.predict_proba(X_test_reduced_scaled)[:,1]

# Reduced Model AUC
reduced_auc = roc_auc_score(y_test_clf, y_prob_reduced)

# ==========================================
# Full Model AUC
# ==========================================

full_auc = roc_auc

# ==========================================
# Comparison
# ==========================================

comparison = pd.DataFrame({
    "Model":[
        "Full Random Forest",
        "Reduced Random Forest"
    ],
    "ROC-AUC":[
        full_auc,
        reduced_auc
    ]
})

print("\nAUC Comparison")
print(comparison)

In [ ]:
# Part 3 - Task 5 - Cross Validated comparison
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import pandas as pd

# -----------------------------------------
# 5-Fold Stratified Cross Validation
# -----------------------------------------
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# -----------------------------------------
# Define Models
# -----------------------------------------
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
}

# -----------------------------------------
# Perform Cross Validation
# -----------------------------------------
results = []

for model_name, model in models.items():

    auc_scores = cross_val_score(
        estimator=model,
        X=X_train_scaled,
        y=y_train_clf,
        cv=cv,
        scoring='roc_auc'
    )

    results.append({
        "Model": model_name,
        "Mean ROC-AUC": auc_scores.mean(),
        "Std ROC-AUC": auc_scores.std()
    })

# -----------------------------------------
# Display Results
# -----------------------------------------
cv_results = pd.DataFrame(results)

print("5-Fold Cross Validation Results")
print(cv_results)

In [ ]:
# Part 3 - Task 6 :Hyperparameter tuning with GridSearchCV:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# -----------------------------------------
# Cross-validation setup
# -----------------------------------------
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# -----------------------------------------
# Build pipeline
# -----------------------------------------
rf_pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

# -----------------------------------------
# Parameter grid
# -----------------------------------------
param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}

# -----------------------------------------
# GridSearchCV
# Use X_train, not X_train_scaled,
# because the pipeline handles imputation and scaling
# -----------------------------------------
grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

# Train grid search
grid_search.fit(X_train, y_train_clf)

# -----------------------------------------
# Best results
# -----------------------------------------
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validated ROC-AUC:")
print(grid_search.best_score_)

In [ ]:
# Part 3 - Task 7 - Manual Learing Curve
from sklearn.metrics import roc_auc_score
import pandas as pd

# -----------------------------------------
# Best pipeline obtained from GridSearchCV
# -----------------------------------------
best_pipeline = grid_search.best_estimator_

# Training fractions
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]

results = []

for fraction in fractions:

    # Number of training samples
    n_samples = int(fraction * len(X_train))

    # Select progressively larger subset
    X_subset = X_train.iloc[:n_samples]
    y_subset = y_train_clf.iloc[:n_samples]

    # Train the best pipeline
    best_pipeline.fit(X_subset, y_subset)

    # -----------------------------
    # Training AUC
    # -----------------------------
    train_prob = best_pipeline.predict_proba(X_subset)[:, 1]
    train_auc = roc_auc_score(y_subset, train_prob)

    # -----------------------------
    # Test AUC
    # -----------------------------
    test_prob = best_pipeline.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test_clf, test_prob)

    # Store results
    results.append({
        "Training Fraction": f"{int(fraction*100)}%",
        "Training AUC": train_auc,
        "Test AUC": test_auc
    })

# -----------------------------------------
# Display Results
# -----------------------------------------
learning_curve = pd.DataFrame(results)

print("Manual Learning Curve")
print(learning_curve)

In [ ]:
# Part 3 - Task 8 - Serialize the best model
best_pipeline = grid_search.best_estimator_

import joblib

# Save the best pipeline
joblib.dump(best_pipeline, "best_model.pkl")

print("Best model saved successfully as 'best_model.pkl'")

import joblib

# Load the saved model
loaded_model = joblib.load("best_model.pkl")

print("Model loaded successfully.")

import pandas as pd

# Two manually created movies
sample_movies = pd.DataFrame([
    {
        "budget": 150000000,
        "runtime": 130,
        "release_year": 2020,
        "release_month": 7,
        "Action": 1,
        "Adventure": 1,
        "Animation": 0,
        "Comedy": 0,
        "Crime": 0,
        "Documentary": 0,
        "Drama": 0,
        "Family": 0,
        "Fantasy": 1,
        "Foreign": 0,
        "History": 0,
        "Horror": 0,
        "Music": 0,
        "Mystery": 0,
        "Romance": 0,
        "Science Fiction": 1,
        "TV Movie": 0,
        "Thriller": 1,
        "War": 0,
        "Western": 0
    },
    {
        "budget": 10000000,
        "runtime": 95,
        "release_year": 2018,
        "release_month": 2,
        "Action": 0,
        "Adventure": 0,
        "Animation": 0,
        "Comedy": 1,
        "Crime": 0,
        "Documentary": 0,
        "Drama": 1,
        "Family": 0,
        "Fantasy": 0,
        "Foreign": 0,
        "History": 0,
        "Horror": 0,
        "Music": 0,
        "Mystery": 0,
        "Romance": 1,
        "Science Fiction": 0,
        "TV Movie": 0,
        "Thriller": 0,
        "War": 0,
        "Western": 0
    }
])

print(sample_movies)

# Predict class labels
predictions = loaded_model.predict(sample_movies)

print("Predictions:")
print(predictions)